![imagen](real_state.png)

# Exercise 3

The objective of this notebook is to compare the performance of a second-degree polynomial model against a standard linear regression model, in order to evaluate which of the two generalizes better to the data.

Throughout the analysis, the following points will be addressed:

1. Polynomial transformation: New features will be generated from the original variables, and the number of features produced by applying a second-degree transformation will be quantified.
2. Model comparison: The performance of both models will be evaluated using error metrics (MSE, MAE, R²) and their generalization ability will be analyzed.
3. Alignment with ASUM-DM: Each section of the notebook will be linked to the phases of the ASUM-DM (Analytics Solutions Unified Method for Data Mining) methodology.

# 1. Import libraries

In [2]:
# System command library
import os
# Data handling libraries
import pandas as pd
# Library for polynomial parameters
from sklearn.preprocessing import PolynomialFeatures
#Library for fitting linear models
from sklearn.linear_model import LinearRegression
# To evaluate model performance using MSE, MAE, and R² metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
# To split the training set into training and test sets.
from sklearn.model_selection import train_test_split

# 2. Load data

In [3]:
data=pd.read_csv('venta_inmuebles_data.csv',sep=',')

In [4]:
data.shape

(5466, 10)

In [5]:
data.head()

,numero_cuartos,numero_baños,area_de_ construcción_pie2,area_del lote_pie2,numero_pisos,condición,grado,superficie_sótano_pie2,año_de_construcción,precio
0,3,1,1180,5650,1,3,7,0,1955,221900
1,2,1,770,10000,1,3,6,0,1933,180000
2,4,3,1960,5000,1,5,7,910,1965,604000
3,3,2,1680,8080,1,3,8,0,1987,510000
4,3,1,1780,7470,1,3,7,730,1960,229500


# 3. Data preparation

In [6]:
data_t = data

In [7]:
# Missing 
data_t.isna().sum()/len(data_t)

numero_cuartos                0.0
numero_baños                  0.0
area_de_ construcción_pie2    0.0
area_del lote_pie2            0.0
numero_pisos                  0.0
condición                     0.0
grado                         0.0
superficie_sótano_pie2        0.0
año_de_construcción           0.0
precio                        0.0
dtype: float64

In [8]:
#Duplicados
data_t.duplicated().sum()

np.int64(1)

In [9]:
data_t[data_t.duplicated()]

,numero_cuartos,numero_baños,area_de_ construcción_pie2,area_del lote_pie2,numero_pisos,condición,grado,superficie_sótano_pie2,año_de_construcción,precio
1193,2,2,1070,649,2,3,9,350,2008,259950


In [10]:
data_t=data_t.drop_duplicates()
data_t.shape

(5465, 10)

# 4. Model Building

### 4.1 Linear model

In [11]:
Y=data_t['precio']
X=data_t.drop(['precio'], axis=1)

In [12]:
X_train, X_test, Y_train, Y_test = train_test_split( X, Y, test_size=0.2, random_state=0)

In [13]:
modelo_regresion = LinearRegression()
modelo_regresion

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [14]:
modelo_regresion.fit(X_train,Y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


#### Linear model evaluation (training data)

In [15]:
y_pred = modelo_regresion.predict(X_train)

print(f"MSE: {mean_squared_error(Y_train, y_pred):.2f}")
print(f"MAE: {mean_absolute_error(Y_train, y_pred):.2f}")
print(f"R²: {r2_score(Y_train, y_pred):.2f}")

MSE: 16003712845.34
MAE: 95901.18
R²: 0.50


### 4.2 Polynomial model

In [16]:
# I use the 2nd degree transformation
poly = PolynomialFeatures(degree=2)
poly_X = poly.fit_transform(X)

In [17]:
X.shape

(5465, 9)

In [18]:
poly_X.shape

(5465, 55)

The transformation generated 46 new variables

In [19]:
poly_X_train, poly_X_test, poly_Y_train, poly_Y_test = train_test_split(poly_X, Y, test_size = 0.2, random_state = 0)

In [20]:
modelo_regresion_poly = LinearRegression().fit(poly_X_train, poly_Y_train)

#### Evaluation of the polynomial model (training data)

In [21]:
y_pred = modelo_regresion_poly.predict(poly_X_train)

print(f"MSE: {mean_squared_error(poly_Y_train, y_pred):.2f}")
print(f"MAE: {mean_absolute_error(poly_Y_train, y_pred):.2f}")
print(f"R²: {r2_score(poly_Y_train, y_pred):.2f}")

MSE: 15303178688.78
MAE: 94135.30
R²: 0.53


# 5. Conclusions

#### Model Performance Comparison

| Metric | Linear Regression | Polynomial Regression (degree 2) |
|--------|-------------------|--------------------------------|
| MSE | 16,003,712,845.34 | 15,303,178,688.78 |
| MAE | 95,901.18 | 94,135.30 |
| R² | 0.50 | 0.53 |

#### Key Findings

1. Both models explain only 50-53% of the price variability (R²), indicating that the current features have moderate explanatory power over property prices.

2. The second-degree polynomial model shows a slight improvement across all metrics:
   - R² increases from 0.50 to 0.53
   - MSE decreases by approximately 4.4%
   - MAE improves by approximately 1.8%

3. Interpretation of MAE: On average, both models make prediction errors of around €94,000-95,900, which is significant given the price range of properties.

#### Recommendations for Improvement

- Consider adding more relevant features (e.g., location-based variables, neighborhood indicators)

#### Final Verdict

The polynomial regression model (degree 2) slightly outperforms the linear baseline, but both models have moderate predictive capacity. 

# 6. ASUM-DM Methodology Alignment

This notebook follows the **ASUM-DM (Analytics Solutions Unified Method for Data Mining)** methodology from IBM:

| Phase | Section |
|-------|---------|
| 1. Business Understanding | Introduction |
| 2. Analytical Approach | Introduction (Linear vs Polynomial) |
| 3. Data Requirements | Data Loading |
| 4. Data Collection | Data Loading |
| 5. Data Understanding | Data Loading (EDA) |
| 6. Data Preparation | Data Preparation |
| 7. Model Building | Model Construction |
| 8. Model Evaluation | Conclusions |